# 36. Agentic RAG

An agent loop that decides when to search, what to search for and when it has enough evidence.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Fixed pipelines do one retrieval and answer. An **agentic RAG** system lets the LLM decide: search or calculate? is the evidence enough? should I search again with a different query? This is the **ReAct loop** - reason, act, observe, repeat - and it can solve multi-step questions that single-shot RAG cannot.

## How it works

1. Give the LLM a prompt describing the available tools
2. Each turn the LLM returns JSON: thought + action + input
3. Run the tool and append the observation to the scratchpad
4. Loop until the LLM chooses `final` or the step limit is hit
5. Log every step for debugging

```text
 question --> [LLM: thought/action] --> tool (search | calculator) --> observation --+
                    ^                                                               |
                    +---------------------------------------------------------------+
                    (repeat until action == final or max_steps)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** Optional: a Contextual AI account if you want to swap in their hosted agent platform

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:15:29.212000 10269 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


Instead of one fixed retrieve-then-generate pass, the LLM runs a **reason -> act -> observe** loop with tools. This notebook uses a local agent; the same loop can be backed by a hosted platform such as Contextual AI (needs an account and API key).

In [2]:
def search_tool(q):
    return "\n".join(f"- {h['text']}" for h in store.search(q, 3))

def calculator_tool(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr):
        return "invalid expression"
    return str(eval(expr, {"__builtins__": {}}))

TOOLS = {"search": search_tool, "calculator": calculator_tool}

AGENT_PROMPT = """You answer questions using tools.
Tools:
- search: look up the handbook. Input: a search query.
- calculator: do arithmetic. Input: an expression.
Reply with ONE JSON object each turn:
{"thought": "...", "action": "search" | "calculator" | "final", "input": "..."}
Use "final" with the answer text as input once you have enough evidence."""

In [3]:
def agent(question, max_steps=5, verbose=True):
    scratch = f"Question: {question}\n"
    for step in range(max_steps):
        move = llm_json(AGENT_PROMPT + "\n\n" + scratch)
        if not isinstance(move, dict) or "action" not in move:
            break
        if verbose: print(f"step {step + 1}: {move.get('thought', '')}\n   -> {move['action']}({move.get('input', '')!r})")
        if move["action"] == "final":
            return move["input"]
        tool = TOOLS.get(move["action"])
        observation = tool(move["input"]) if tool else "unknown tool"
        scratch += f"Action: {move['action']}({move['input']})\nObservation: {observation}\n"
    return llm(scratch + "\nGive the best final answer from the observations above.")

print(agent("How many dimensions does all-MiniLM-L6-v2 produce, and how many floats is that for 1000 documents?"))

step 1: All-MiniLM-L6-v2 is a transformer-based language model
   -> calculator('1000 * (get dimensions of all-MiniLM-L6-v2)')


step 2: Invalid expression syntax
   -> calculator('1000 * (get dimensions of all-MiniLM-L6-v2)')


step 3: Invalid input for calculator. Expression should be a valid arithmetic operation.
   -> calculator('1000 * (get dimensions of all-MiniLM-L6-v2)')


step 4: The get dimensions of all-MiniLM-L6-v2 command is not a valid input for the calculator.
   -> search('all-MiniLM-L6-v2 dimensions')


step 5: all-MiniLM-L6-v2 produces 384-dimensional vectors
   -> calculator('1000 * 384')


The all-MiniLM-L6-v2 model produces 384-dimensional vectors.

To calculate the total number of floats for 1000 documents, we multiply the dimensionality (384) by the number of documents (1000):

384 * 1000 = 384,000

So, the final answer is: The all-MiniLM-L6-v2 model produces 384-dimensional vectors, and for 1000 documents, it would produce approximately 384,000 floats.


**Trade-off:** more flexible and able to do multi-hop research, but costs more tokens, adds latency and needs step limits plus logging to stay debuggable.

## Trace a multi-step question
This question needs a search *and* a calculation, which single-shot RAG cannot do.

In [4]:
q = "What is the Reciprocal Rank Fusion constant, and what is 1 divided by that constant plus 5 as a decimal?"
print("FINAL:", agent(q))

step 1: Reciprocal Rank Fusion constant
   -> search('Reciprocal Rank Fusion constant')


step 2: Reciprocal Rank Fusion constant is 60
   -> calculator('1 / (60 + 5)')


step 3: Reciprocal Rank Fusion constant is 60
   -> final('1 / (60 + 5) = 0.015384615384615385')
FINAL: 1 / (60 + 5) = 0.015384615384615385


## Safety rails
Agents need limits. Our loop has a step cap and the calculator only accepts arithmetic characters.

In [5]:
print(calculator_tool("2+2*3"))
print(calculator_tool("__import__('os').system('ls')"))

8
invalid expression


## Cost of agency
Count LLM calls (steps) for simple vs complex questions.

In [6]:
import time
for q in ["What is HNSW?", "How many dimensions does all-MiniLM-L6-v2 produce, and how many floats for 500 documents?"]:
    t = time.time(); out = agent(q, verbose=False); print(f"{time.time() - t:5.1f}s  {q[:70]}\n      -> {out[:150]}")

 24.0s  What is HNSW?
      -> Based on the observations, HNSW stands for Hierarchical Navigable Small World, which is a data structure used in knowledge graph-based systems to effi


 11.6s  How many dimensions does all-MiniLM-L6-v2 produce, and how many floats
      -> {"dimensions": 384, "floats": 500 * 4}


## Compare against plain RAG

In [7]:
compare_pipelines({"plain RAG": lambda q: rag(q, store, 3)[0], "agentic RAG": lambda q: agent(q, verbose=False)}, n=4)

,keyword_recall,judge_correct,avg_seconds
pipeline,,,
plain RAG,0.79,1.0,0.0
agentic RAG,0.50,0.5,9.0


## Using Contextual AI
If you have a Contextual AI account you can replace the local `TOOLS` with their hosted retrieval/agent APIs. Keep the same loop structure: the agent decides, the platform retrieves. Check their documentation for the current SDK and endpoints.

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Handles multi-step and multi-tool questions | More latency and tokens |
| Can recover by re-querying | Small models often produce invalid JSON or loop |
| Extensible: add SQL, web, calculators | Harder to test and debug |

## When to use it

Complex questions requiring several lookups or tools. Start with a fixed pipeline; add agency only where needed.

## Experiments to try

- Add a `lookup_csv` tool using the products table.
- Detect repeated identical actions and break the loop.
- Parse tool calls with Ollama's native tool-calling API.

## Key takeaways

- Reason -> act -> observe.
- Always cap steps.
- Log the trace.

---
**Next:** [37. Evaluation with GroUSE-style Judges](../37_evaluation_grouse/evaluation_grouse.ipynb)